# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata  # Keep as a single object (do not treat as a dict)

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. We'll look at all available record sets and fields, referencing their `@id`.

In [ ]:
# Show all RecordSets and their fields with @id

print("Available Record Sets:")
for record_set in dataset.record_sets:
    print(f"- RecordSet @id: {record_set['@id']}")
    print(f"  Name: {record_set.get('name','(unnamed)')}")
    print("  Fields (with @id and name):")
    for field in record_set.get('field', []):
        field_obj = field if isinstance(field, dict) else dataset.lookup(field)
        print(f"    - @id: {field_obj['@id']}, name: {field_obj.get('name','(unnamed)')}")
    print()
# For demonstration, get the list of all record set ids:
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
print(f"Discovered record set ids: {record_set_ids}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from all record sets
dataframes = {}

for record_set_id in record_set_ids:
    try:
        # Load all records for this record set
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for record set {record_set_id} with {len(df)} rows and columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"Failed to load record set {record_set_id}: {e}")

# For further analysis, pick the first available record set as example
if record_set_ids:
    first_rs = record_set_ids[0]
    print(f"Available columns in record set '{first_rs}':")
    print(dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 
We'll use example columns for demonstration; adjust column `@id` variables as needed based on the real schema content.

In [ ]:
# Choose one record set to analyze
record_set_id = first_rs
df = dataframes[record_set_id]

# Display all available columns for selection
print(f"Columns in the DataFrame: {df.columns.tolist()}")
# Select an example numeric field (by its field @id or column name)
# Common field names: 'abundance', 'peptide_count', 'molecular_weight' etc. -- adjust below if necessary

example_numeric_candidates = [col for col in df.columns if any(x in col.lower() for x in ["abundance","count","weight","coverage","mw","intensity"])]
if example_numeric_candidates:
    numeric_field = example_numeric_candidates[0]
else:
    numeric_field = df.select_dtypes('number').columns[0]  # fallback

print(f"Using numeric field: {numeric_field}")

threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
filtered_df = df[df[numeric_field] > threshold]

print(f"Filtered records with {numeric_field} > {threshold:.3f} (mean):")
print(filtered_df.head())

filtered_df = filtered_df.copy() # Avoid pandas SettingWithCopyWarning
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a candidate categorical field, e.g. a 'description' or 'accession'
example_group_candidates = [col for col in df.columns if any(x in col.lower() for x in ["sample","type","class","description","accession"]) and col!=numeric_field]
if example_group_candidates:
    group_field = example_group_candidates[0]
    print(f"Grouping by: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Mean of {numeric_field} grouped by {group_field} (first five groups):")
    print(grouped_df.head())
else:
    print("No suitable group field detected.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We'll show a histogram and, if possible, a group-by bar plot.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If we found a group field above, plot group mean
if 'grouped_df' in locals():
    plt.figure(figsize=(10,4))
    plot_df = grouped_df.sort_values(numeric_field, ascending=False).head(20)
    sns.barplot(data=plot_df, x=group_field, y=numeric_field)
    plt.title(f"Mean {numeric_field} by {group_field} (top 20 groups)")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to use the `mlcroissant` library to load mass spectrometry protein abundance data in the Croissant format, explored the dataset structure, extracted records by RecordSet `@id`, and performed basic EDA and visualizations. Referencing fields and sets by `@id` enables robust downstream processing across all Croissant-compliant datasets.

Next steps might include applying more advanced analyses, integrating annotation metadata, or exporting processed subsets for bioinformatics pipelines.